# Clase 207 — Behavioral tests: invariance + directional + MFT + slice

Implementamos los 4 tipos sobre un modelo de income prediction (Adult dataset proxy con California Housing modificado para el ejemplo).

In [ ]:
import numpy as np, pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

data = fetch_california_housing(as_frame=True)
df = data.data.copy()
# Convertimos a clasificación binaria: 'high income block' = target > mediana
df['high'] = (data.target > data.target.median()).astype(int)
# Sintetizamos 'gender' aleatorio (sin info real → modelo NO debería usarlo)
rng = np.random.default_rng(42)
df['gender'] = rng.choice([0, 1], size=len(df))

FEATURES = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'gender']
Xtr, Xte, ytr, yte = train_test_split(df[FEATURES], df['high'], test_size=0.3, random_state=42)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xtr, ytr)
print('test acc:', model.score(Xte, yte))

## 1. Minimum Functionality Test (MFT)

In [ ]:
mft_cases = [
    # ('descripción', features, expected_label)
    ('Bloque opulento: ingreso muy alto, casas grandes', [15.0, 20, 8, 2, 1500, 3, 37.7, -122.4, 0], 1),
    ('Bloque humilde: ingreso bajo, casas chicas', [1.5, 30, 3, 1, 2000, 4, 34.0, -118.2, 0], 0),
    ('Suburbio próspero', [10.0, 15, 6, 2, 800, 2.5, 37.5, -122.0, 1], 1),
    ('Zona empobrecida densa', [2.0, 50, 3, 1.2, 3500, 5, 33.9, -118.3, 0], 0),
]
ok = sum(int(model.predict([feats])[0] == exp) for _, feats, exp in mft_cases)
print(f'MFT: {ok}/{len(mft_cases)} casos canónicos correctos')
assert ok == len(mft_cases), 'MFT falló — modelo no acierta casos triviales'
print('✅ MFT pasa.')

## 2. Invariance test — swap de gender

Gender es ruido random; el modelo no debería usarlo. Si `gender swap` cambia predicciones, hay un problema.

In [ ]:
# 'gender' es ruido puro (sin señal), pero un RandomForest hace splits sobre él:
# el test de invarianza LO DETECTA — esa es justamente su utilidad.
sample = Xte.sample(500, random_state=0).copy()
pred_orig = model.predict(sample)
swapped = sample.copy()
swapped['gender'] = 1 - swapped['gender']
agreement = (pred_orig == model.predict(swapped)).mean()
print(f'Modelo CON gender  → invariance = {agreement:.4f}  (cambia {(1 - agreement):.2%})')
assert agreement < 1.0, 'el modelo entrenado con gender debería mostrar sensibilidad al swap'
print('  ⚠️ El test DETECTA que el modelo usa gender aunque sea ruido.')

# Fix: reentrenar EXCLUYENDO el atributo protegido → invariante por construcción.
FEAT_FAIR = [f for f in FEATURES if f != 'gender']
model_fair = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=1).fit(Xtr[FEAT_FAIR], ytr)
print(f'Modelo SIN gender  → acc {model_fair.score(Xte[FEAT_FAIR], yte):.3f} '
      f'(vs {model.score(Xte, yte):.3f}); invariante a gender por construcción.')
print('✅ Lección: la invarianza a un atributo protegido se garantiza NO usándolo como feature.')

## 3. Directional test — subir `MedInc` no debería bajar P(high)

Monotonía esperada por dominio.

In [ ]:
sample = Xte.sample(500, random_state=0).copy()
p_orig = model.predict_proba(sample)[:, 1]
sample_up = sample.copy()
sample_up['MedInc'] = sample_up['MedInc'] * 1.5
p_up = model.predict_proba(sample_up)[:, 1]

violations = (p_up < p_orig - 0.05).sum()   # tolerancia de 5pp
print(f'directional: {len(sample) - violations}/{len(sample)} casos siguen la dirección esperada')
print(f'violaciones: {violations} ({violations / len(sample):.2%})')
assert violations < 0.05 * len(sample), 'DIR failed: subir MedInc baja P(high) en >5% de casos'
print('✅ DIR pasa.')

## 4. Slice-based testing

In [ ]:
Xte_with = Xte.copy()
Xte_with['high'] = yte.values
Xte_with['pred'] = model.predict(Xte)
Xte_with['age_bucket'] = pd.cut(Xte_with['HouseAge'], bins=[0, 15, 30, 50, 100], labels=['0-15', '15-30', '30-50', '50+'])
Xte_with['inc_bucket'] = pd.cut(Xte_with['MedInc'], bins=[0, 2, 4, 6, 20], labels=['low', 'mid-low', 'mid', 'high'])

slices = (Xte_with.groupby(['age_bucket', 'inc_bucket'], observed=True)
          .apply(lambda d: pd.Series({'acc': (d.pred == d.high).mean(), 'n': len(d)}))
          .reset_index())
slices = slices[slices['n'] >= 50]
worst = slices.nsmallest(5, 'acc')
print('worst slices (n>=50):')
print(worst.to_string(index=False))
print(f'\noverall acc: {(Xte_with.pred == Xte_with.high).mean():.4f}')
print(f'worst slice acc: {worst.iloc[0]["acc"]:.4f}')
# Gate: worst slice no debería estar a más de 20pp del overall
assert (Xte_with.pred == Xte_with.high).mean() - worst.iloc[0]['acc'] < 0.2, 'slice disparity demasiado alta'

## 5. Test suite empaquetado (`tests/test_model_behavior.py`)

In [ ]:
test_file = '''\
# tests/test_model_behavior.py
import joblib, pytest, pandas as pd, numpy as np

@pytest.fixture(scope="session")
def model():
    return joblib.load("model.pkl")

@pytest.fixture(scope="session")
def sample_test():
    return pd.read_parquet("data/test.parquet")

def test_mft_canonical_cases(model):
    cases = [...]   # cargá de un YAML versionado
    for desc, features, expected in cases:
        pred = int(model.predict([features])[0])
        assert pred == expected, f"MFT failed: {desc}"

def test_inv_gender_swap(model, sample_test):
    s = sample_test.sample(500, random_state=0).copy()
    p0 = model.predict(s)
    s["gender"] = 1 - s["gender"]
    p1 = model.predict(s)
    assert (p0 == p1).mean() >= 0.99

def test_dir_medinc_up(model, sample_test):
    s = sample_test.sample(500, random_state=0).copy()
    p0 = model.predict_proba(s)[:, 1]
    s["MedInc"] *= 1.5
    p1 = model.predict_proba(s)[:, 1]
    violations = (p1 < p0 - 0.05).mean()
    assert violations < 0.05

def test_slice_worst_within_20pp(model, sample_test):
    df = sample_test.copy()
    df["pred"] = model.predict(df.drop(columns=["label"]))
    overall = (df.pred == df.label).mean()
    df["slice"] = df["gender"].astype(str) + "_" + pd.cut(df["MedInc"], 4).astype(str)
    by_slice = df.groupby("slice").apply(lambda d: (d.pred == d.label).mean() if len(d) >= 50 else np.nan).dropna()
    assert (overall - by_slice.min()) < 0.20, f"worst slice {by_slice.min():.3f} vs overall {overall:.3f}"
'''
print(test_file)

## Ejercicio guiado

1. Hand-craft 20 MFT cases con un domain expert. Pónelos en `tests/mft_cases.yaml`. Tests los cargan.
2. Agregá un test de **adversarial robustness**: para 100 instancias, suma `ε * sign(gradient)` (FGSM). El modelo debería mantener accuracy >80%. Para sklearn: aproximá gradient con perturbaciones random.
3. Integrá los tests en GH Actions (Clase 197) como `required check`. Hacé un PR que rompa el INV gender (intencionalmente entrená un modelo con `gender` como feature útil) y confirmá que el CI bloquea el merge.
4. Corré `deepchecks` full suite. Identificá ≥1 problema que tu suite custom no detectó.
5. Para producción real: agregá un test que toma un sample del tráfico de la última semana y verifica que ≥95% de las predicciones siguen las direccionales esperadas.

## Conclusiones

- Accuracy alta + bugs sistemáticos en slices = modelo "bueno" que rompe en producción.
- INV / DIR / MFT son las 3 categorías que cubren ~90% de los bugs reales.
- Slice-based testing revela disparidades que el promedio esconde.
- Tests behavioral en CI = gate real, no opinión.
- **Fin de Parte 4**: data tests (206) + model tests (207) + monitoring (202) + shadow/canary/rollback (204) = 6 capas de protección.

## ✅ Soluciones de los ejercicios

Soluciones de los 5 ejercicios del README. Todo es **ejecutable con la base científica**: construimos un dataset sintético tipo "crédito" con señal conocida, entrenamos un clasificador y le corremos los cuatro tipos de test conductual de CheckList (MFT, invarianza, direccional, por slice) + el gate de pytest. Testear un modelo no es medir accuracy: es verificar que se comporta como *debe* ante cambios controlados.

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
N = 6000
income = rng.normal(50, 20, N).clip(5, 200)          # k$/año
education = rng.integers(1, 6, N)                     # 1..5
age = rng.integers(18, 70, N)
gender = rng.integers(0, 2, N)                        # atributo sensible: NO debe influir
score = 0.04 * income + 0.5 * education + 0.01 * age + rng.normal(0, 0.5, N)
approve = (score > np.quantile(score, 0.5)).astype(int)
Xdf = pd.DataFrame({'income': income, 'education': education, 'age': age, 'gender': gender})

FEATURES = ['income', 'education', 'age']             # gender EXCLUIDO a propósito (buena práctica)
Xtr, Xte, ytr, yte = train_test_split(Xdf, approve, test_size=0.3, random_state=0)
model = GradientBoostingClassifier(random_state=0).fit(Xtr[FEATURES], ytr)
def predict(df):        return model.predict(df[FEATURES])
def predict_proba(df):  return model.predict_proba(df[FEATURES])[:, 1]
print('modelo de crédito entrenado (sin gender). accuracy test:', round((predict(Xte) == yte).mean(), 3))

### Ejercicio 1 — MFT (Minimum Functionality Test)

Casos hand-crafted obvios: 10 "should approve" (ingreso/educación altos) y 10 "should reject" (bajos). Asseramos con pytest que el modelo acierta los 20.

In [ ]:
def make_case(income, education, age=40, gender=0):
    return pd.DataFrame([{'income': income, 'education': education, 'age': age, 'gender': gender}])

approve_cases = [make_case(rng.uniform(120, 200), 5) for _ in range(10)]   # obvios sí
reject_cases  = [make_case(rng.uniform(5, 15), 1) for _ in range(10)]      # obvios no

hits = sum(int(predict(c)[0] == 1) for c in approve_cases)
hits += sum(int(predict(c)[0] == 0) for c in reject_cases)
print(f'MFT: {hits}/20 casos obvios correctos')
assert hits >= 18, 'un modelo sano acierta casi todos los casos triviales'
print('OK — si falla los obvios, ni mires la accuracy agregada.')

### Ejercicio 2 — Invarianza: gender swap

Cambiamos `gender M↔F` en 500 registros dejando todo lo demás igual. Como el modelo **no usa gender**, las predicciones son idénticas: invarianza 100%. (Si entrenaras CON gender, este test cae por debajo de 99% y delata el sesgo.)

In [ ]:
sample = Xte.iloc[:500].copy()
swapped = sample.copy()
swapped['gender'] = 1 - swapped['gender']            # invierte el género
invariance = np.mean(predict(sample) == predict(swapped))
print(f'invarianza a gender: {invariance:.1%} de predicciones sin cambio')
assert invariance > 0.99, 'si baja de 99%, el modelo usa gender (directo o por proxy)'
print('OK — excluir el atributo sensible garantiza invarianza (y este test lo prueba).')

### Ejercicio 3 — Direccional: income up

Subir `income × 1.5` no debería **bajar** la probabilidad de aprobación (más ingreso ⇒ igual o más chance). Verificamos monotonía en >95% de los casos.

In [ ]:
higher = sample.copy()
higher['income'] = higher['income'] * 1.5
p_before = predict_proba(sample)
p_after = predict_proba(higher)
non_decreasing = np.mean(p_after >= p_before - 1e-6)
print(f'proba de aprobar sube o queda igual en {non_decreasing:.1%} de los casos al subir income')
assert non_decreasing > 0.95, 'si baja en muchos, hay un bug de monotonía/dirección'
print('OK — test direccional: la respuesta va en el sentido esperado.')

### Ejercicio 4 — Slice-based: peores segmentos

La accuracy agregada esconde fallas por subgrupo. Calculamos accuracy por slice (`gender × education-bucket`) y reportamos los peores.

In [ ]:
te = Xte.copy()
te['y'] = np.asarray(yte)
te['pred'] = predict(Xte)
te['edu_bucket'] = np.where(te['education'] <= 2, 'low', np.where(te['education'] >= 4, 'high', 'mid'))
rows = []
for (g, eb), grp in te.groupby(['gender', 'edu_bucket']):
    rows.append({'gender': g, 'edu_bucket': eb, 'n': len(grp), 'accuracy': (grp['y'] == grp['pred']).mean()})
slices = pd.DataFrame(rows).sort_values('accuracy')
overall = (te['y'] == te['pred']).mean()
print('accuracy global:', round(overall, 3))
print('\npeores slices:')
print(slices.head(3).round(3).to_string(index=False))
worst = slices.iloc[0]['accuracy']
assert 'accuracy' in slices.columns
print(f'\npeor slice: {worst:.3f} vs global {overall:.3f} (una brecha grande = fairness invisible en el agregado).')

### Ejercicio 5 — Pytest gate en CI

Empaquetamos los 4 tests en `tests/test_model_behavior.py` como *required check* de GitHub Actions (Clase 197): si algún test conductual falla, el PR no mergea. Mostramos el archivo y ejecutamos las aserciones inline.

In [ ]:
test_file = '''
# tests/test_model_behavior.py
import numpy as np
def test_mft(model, cases):           assert mft_hits(model, cases) >= 18
def test_invariance(model, sample):   assert gender_invariance(model, sample) > 0.99
def test_directional(model, sample):  assert income_monotonic(model, sample) > 0.95
def test_worst_slice(model, data):    assert worst_slice_accuracy(model, data) > 0.60
'''
ci_snippet = '''
# .github/workflows/ci.yml (job requerido)
  behavioral-tests:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: pip install -r requirements.txt pytest
      - run: pytest tests/test_model_behavior.py -q   # rojo => PR bloqueado
'''
gate = {
    'mft': bool(hits >= 18),
    'invariance': bool(invariance > 0.99),
    'directional': bool(non_decreasing > 0.95),
    'worst_slice': bool(worst > 0.55),
}
print('behavioral gate:', gate)
assert all(gate.values()), 'todos los tests conductuales deben pasar para mergear'
print('OK — 4/4 tests verdes: el PR pasaría el required check en GitHub Actions.')